In [0]:
%sql
use catalog deltalake_catalog;

In [0]:

%sql
DROP TABLE IF EXISTS orders_managed;

CREATE OR REPLACE TABLE orders_managed (
  order_id BIGINT,
  sku STRING,
  product_name STRING,
  product_category STRING,
  qty INT,
  unit_price DECIMAL(10,2)
)
USING DELTA
TBLPROPERTIES (
  delta.autoOptimize.optimizeWrite = false,
  delta.autoOptimize.autoCompact = false
)


In [0]:
%sql
describe detail orders_managed;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,9b7e47af-96f4-49ec-b33f-c2bec3add8fa,deltalake_catalog.default.orders_managed,null,abfss://unitycatalog@ttmystorageaccount001.dfs.core.windows.net/catalog/__unitystorage/catalogs/758280e5-e8ae-48b0-840c-9f10d52cc821/tables/5a2296fc-9280-4690-a58b-89a40397b9d4,2025-09-26T10:37:14.409Z,2025-09-26T10:37:14.000Z,List(),List(),0,0,"Map(delta.autoOptimize.autoCompact -> false, collation -> UTF8_BINARY, delta.autoOptimize.optimizeWrite -> false, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import DecimalType
from pyspark.sql.functions import col
import time


N = 100

for i in range(N):
    row = Row(order_id=int(i+1),
              sku=f"SKU-{(i%10)+1}",
              product_name=f"Product-{(i%50)+1}",
              product_category=f"Category-{(i%5)+1}",
              qty=(i % 10) + 1,
              unit_price=round(10.0 + (i%7)*1.5, 2))
    
    df = spark.createDataFrame([row])

    df = df.withColumn("qty", col("qty").cast("int")).withColumn("unit_price", col("unit_price").cast(DecimalType(10,2)))
    
    df.write.format("delta").mode("append").saveAsTable("orders_managed")
    # optional tiny sleep to mimic real small-batch writes
    # time.sleep(0.02)

print(f"Inserted {N} tiny writes.")

Inserted 100 tiny writes.


In [0]:
%sql
select * from orders_managed;

order_id,sku,product_name,product_category,qty,unit_price
20,SKU-10,Product-20,Category-5,10,17.50
40,SKU-10,Product-40,Category-5,10,16.00
70,SKU-10,Product-20,Category-5,10,19.00
100,SKU-10,Product-50,Category-5,10,11.50
90,SKU-10,Product-40,Category-5,10,17.50
50,SKU-10,Product-50,Category-5,10,10.00
80,SKU-10,Product-30,Category-5,10,13.00
30,SKU-10,Product-30,Category-5,10,11.50
60,SKU-10,Product-10,Category-5,10,14.50
10,SKU-10,Product-10,Category-5,10,13.00


In [0]:
%sql
select * from orders_managed where order_id = 56;

order_id,sku,product_name,product_category,qty,unit_price
56,SKU-6,Product-6,Category-1,6,19.00


In [0]:
query = "select AVG(unit_price) as avg_price from orders_managed"
res = spark.sql(query).collect()
print(res)

[Row(avg_price=Decimal('14.425000'))]


In [0]:
%sql
DESCRIBE HISTORY orders_managed;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
101,2025-09-26T10:46:48.000Z,147711536460494,sumit.trendytech03@outlook.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2999395735835751),0926-100404-x5jzm2ub-v2n,100,SnapshotIsolation,false,"Map(numRemovedFiles -> 100, numRemovedBytes -> 172026, p25FileSize -> 3178, numDeletionVectorsRemoved -> 0, minFileSize -> 3178, numAddedFiles -> 1, maxFileSize -> 3178, p75FileSize -> 3178, p50FileSize -> 3178, numAddedBytes -> 3178)",null,Databricks-Runtime/17.1.x-photon-scala2.13
100,2025-09-26T10:40:48.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835751),0926-100404-x5jzm2ub-v2n,99,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1726)",null,Databricks-Runtime/17.1.x-photon-scala2.13
99,2025-09-26T10:40:46.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835751),0926-100404-x5jzm2ub-v2n,98,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1721)",null,Databricks-Runtime/17.1.x-photon-scala2.13
98,2025-09-26T10:40:45.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835751),0926-100404-x5jzm2ub-v2n,97,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1720)",null,Databricks-Runtime/17.1.x-photon-scala2.13
97,2025-09-26T10:40:44.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835751),0926-100404-x5jzm2ub-v2n,96,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1721)",null,Databricks-Runtime/17.1.x-photon-scala2.13
96,2025-09-26T10:40:43.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835751),0926-100404-x5jzm2ub-v2n,95,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1721)",null,Databricks-Runtime/17.1.x-photon-scala2.13
95,2025-09-26T10:40:42.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835751),0926-100404-x5jzm2ub-v2n,94,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1720)",null,Databricks-Runtime/17.1.x-photon-scala2.13
94,2025-09-26T10:40:40.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835751),0926-100404-x5jzm2ub-v2n,93,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1721)",null,Databricks-Runtime/17.1.x-photon-scala2.13
93,2025-09-26T10:40:39.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835751),0926-100404-x5jzm2ub-v2n,92,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1721)",null,Databricks-Runtime/17.1.x-photon-scala2.13
92,2025-09-26T10:40:38.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2999395735835751),0926-100404-x5jzm2ub-v2n,91,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1721)",null,Databricks-Runtime/17.1.x-photon-scala2.13


In [0]:
%sql
OPTIMIZE orders_managed;

path,metrics
abfss://unitycatalog@ttmystorageaccount001.dfs.core.windows.net/catalog/__unitystorage/catalogs/758280e5-e8ae-48b0-840c-9f10d52cc821/tables/5a2296fc-9280-4690-a58b-89a40397b9d4,"List(1, 100, List(3178, 3178, 3178.0, 1, 3178), List(1714, 1726, 1720.26, 100, 172026), 0, null, null, 0, 1, 100, 0, true, 0, 0, 1758883605457, 1758883608256, 8, 1, null, List(0, 0), null, 6, 6, 203, 0, null)"


In [0]:
query = "select AVG(unit_price) as avg_price from orders_managed"
res = spark.sql(query).collect()
print(res)

[Row(avg_price=Decimal('14.425000'))]
